### Dataset

In this homework, we'll build a model for predicting if we have an image of a bee or a wasp. 
For this, we will use the "Bee or Wasp?" dataset that was obtained from [Kaggle](https://www.kaggle.com/datasets/jerzydziewierz/bee-vs-wasp) and slightly rebuilt. 

You can download the dataset for this homework from [here](https://github.com/SVizor42/ML_Zoomcamp/releases/download/bee-wasp-data/data.zip):

```bash
wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/bee-wasp-data/data.zip
unzip data.zip
```

In the lectures we saw how to use a pre-trained neural network. In the homework, we'll train a much smaller model from scratch. 

> **Note:** you will need an environment with a GPU for this homework. We recommend to use [Saturn Cloud](https://bit.ly/saturn-mlzoomcamp). 
> You can also use a computer without a GPU (e.g. your laptop), but it will be slower.

In [23]:
import numpy as np

In [24]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


Num GPUs Available:  1


In [25]:
from tensorflow import keras

### Model

For this homework we will use Convolutional Neural Network (CNN). Like in the lectures, we'll use Keras.

You need to develop the model with following structure:

* The shape for input should be `(150, 150, 3)`
* Next, create a convolutional layer ([`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/)):
    * Use 32 filters
    * Kernel size should be `(3, 3)` (that's the size of the filter)
    * Use `'relu'` as activation 
* Reduce the size of the feature map with max pooling ([`MaxPooling2D`](https://keras.io/api/layers/pooling_layers/max_pooling2d/))
    * Set the pooling size to `(2, 2)`
* Turn the multi-dimensional result into vectors using a [`Flatten`](https://keras.io/api/layers/reshaping_layers/flatten/) layer
* Next, add a `Dense` layer with 64 neurons and `'relu'` activation
* Finally, create the `Dense` layer with 1 neuron - this will be the output
    * The output layer should have an activation - use the appropriate activation for the binary classification case

As optimizer use [`SGD`](https://keras.io/api/optimizers/sgd/) with the following parameters:

* `SGD(lr=0.002, momentum=0.8)`

For clarification about kernel size and max pooling, check [Office Hours](https://www.youtube.com/watch?v=1WRgdBTUaAc).

In [26]:
model = keras.Sequential()
model.add(keras.layers.Input((150, 150, 3)))
model.add(keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'))
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(units=64, activation='relu'))
model.add(keras.layers.Dense(units=1, activation="sigmoid"))


optmzr = keras.optimizers.legacy.SGD(
    learning_rate=0.002,
    momentum=0.8
    
)          
model.compile(optimizer=optmzr, loss='binary_crossentropy', metrics=['accuracy'])       

### Question 1

Since we have a binary classification problem, what is the best loss function for us?

* `mean squared error`
* `binary crossentropy`
* `categorical crossentropy`
* `cosine similarity`

> **Note:** since we specify an activation for the output layer, we don't need to set `from_logits=True`

answer: binary crossentropy

### Question 2

What's the number of parameters in the convolutional layer of our model? You can use the `summary` method for that. 

* 1 
* 65
* 896
* 11214912

In [27]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_1 (Conv2D)           (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 74, 74, 32)        0         
 g2D)                                                            
                                                                 
 flatten_1 (Flatten)         (None, 175232)            0         
                                                                 
 dense_2 (Dense)             (None, 64)                11214912  
                                                                 
 dense_3 (Dense)             (None, 1)                 65        
                                                                 
Total params: 11215873 (42.79 MB)
Trainable params: 11215873 (42.79 MB)
Non-trainable params: 0 (0.00 Byte)
____________

answer: 896

### Generators and Training

For the next two questions, use the following data generator for both train and test sets:

```python
ImageDataGenerator(rescale=1./255)
```

* We don't need to do any additional pre-processing for the images.
* When reading the data from train/test directories, check the `class_mode` parameter. Which value should it be for a binary classification problem?
* Use `batch_size=20`
* Use `shuffle=True` for both training and test sets. 

For training use `.fit()` with the following params:

```python
model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)
```

In [28]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [29]:
tf.random.set_seed(1)

In [30]:
train_gen = ImageDataGenerator(rescale=1./255)

In [31]:
train_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory(
    './data/train',
    class_mode='binary',
    target_size=(150, 150),
    batch_size=20,
    shuffle=True,
    seed=1


)

test_gen = ImageDataGenerator(rescale=1./255)

test_ds = test_gen.flow_from_directory(
    './data/test',
    class_mode='binary',
    target_size=(150, 150),
    batch_size=20,
    shuffle=True,
    seed=1
)

Found 3677 images belonging to 2 classes.
Found 918 images belonging to 2 classes.


In [32]:
train_ds.class_indices

{'bee': 0, 'wasp': 1}

In [33]:
test_ds.class_indices

{'bee': 0, 'wasp': 1}

In [34]:
history = model.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds
)

Epoch 1/10
  1/184 [..............................] - ETA: 45s - loss: 0.7411 - accuracy: 0.3500

2024-10-07 08:33:50.257387: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


183/184 [============================>.] - ETA: 0s - loss: 0.6943 - accuracy: 0.5469

2024-10-07 08:33:57.327914: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


184/184 [==============================] - 8s 42ms/step - loss: 0.6944 - accuracy: 0.5466 - val_loss: 0.6229 - val_accuracy: 0.6514
Epoch 2/10
184/184 [==============================] - 8s 42ms/step - loss: 0.6255 - accuracy: 0.6435 - val_loss: 0.5851 - val_accuracy: 0.6939
Epoch 3/10
184/184 [==============================] - 8s 42ms/step - loss: 0.5702 - accuracy: 0.7095 - val_loss: 0.5700 - val_accuracy: 0.7037
Epoch 4/10
184/184 [==============================] - 8s 41ms/step - loss: 0.5339 - accuracy: 0.7460 - val_loss: 0.5634 - val_accuracy: 0.7070
Epoch 5/10
184/184 [==============================] - 8s 42ms/step - loss: 0.5194 - accuracy: 0.7539 - val_loss: 0.5332 - val_accuracy: 0.7397
Epoch 6/10
184/184 [==============================] - 8s 42ms/step - loss: 0.4980 - accuracy: 0.7699 - val_loss: 0.6370 - val_accuracy: 0.6296
Epoch 7/10
184/184 [==============================] - 8s 42ms/step - loss: 0.4905 - accuracy: 0.7710 - val_loss: 0.5471 - val_accuracy: 0.7309
Epoch 8/10

### Question 3

What is the median of training accuracy for all the epochs for this model?

* 0.20
* 0.40
* 0.60
* 0.80

In [35]:
history.history.keys()

dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])

In [36]:
np.median(history.history['accuracy'])

0.7618982791900635

answer: I guess the closest is 0.80, I ran it twice and both times it was a little inbetween the last two options but clightly closer to 0.80

actually, I had been using incorrect code where where I hadnt changed someting that I copied and pasted in the generator/dataset block.  Now it appears that the closest is 0.60

I ran it again and the result is closer to 0.80 again

### Question 4

What is the standard deviation of training loss for all the epochs for this model?

* 0.031
* 0.061
* 0.091
* 0.131

In [37]:
np.std(history.history['loss'])

0.07595731369490528

answer: closest is 0.061

### Data Augmentation

For the next two questions, we'll generate more data using data augmentations. 

Add the following augmentations to your training data generator:

* `rotation_range=50,`
* `width_shift_range=0.1,`
* `height_shift_range=0.1,`
* `zoom_range=0.1,`
* `horizontal_flip=True,`
* `fill_mode='nearest'`

In [38]:
train_gen_augmented = ImageDataGenerator(rescale=1./255,
                              rotation_range=50,
                              width_shift_range=0.1,
                              height_shift_range=0.1,
                              zoom_range=0.1,
                              horizontal_flip=True,
                              fill_mode='nearest')

train_ds_augmented = train_gen_augmented.flow_from_directory(
    './data/train',
    class_mode='binary',
    target_size=(150, 150),
    batch_size=20,
    shuffle=True,
    seed=1


)

test_gen = ImageDataGenerator(rescale=1./255)

test_ds = test_gen.flow_from_directory(
    './data/test',
    class_mode='binary',
    target_size=(150, 150),
    batch_size=20,
    shuffle=True,
    seed=1
)

Found 3677 images belonging to 2 classes.
Found 918 images belonging to 2 classes.


### Question 5 

Let's train our model for 10 more epochs using the same code as previously.
> **Note:** make sure you don't re-create the model - we want to continue training the model
we already started training.

What is the mean of test loss for all the epochs for the model trained with augmentations?

* 0.18
* 0.48
* 0.78
* 0.108

In [39]:
history2 = model.fit(
    train_ds_augmented,
    epochs=10,
    validation_data=test_ds
)

Epoch 1/10
184/184 [==============================] - 12s 66ms/step - loss: 0.5573 - accuracy: 0.7280 - val_loss: 0.5876 - val_accuracy: 0.7081
Epoch 2/10
184/184 [==============================] - 12s 66ms/step - loss: 0.5577 - accuracy: 0.7310 - val_loss: 0.5379 - val_accuracy: 0.7440
Epoch 3/10
184/184 [==============================] - 12s 66ms/step - loss: 0.5786 - accuracy: 0.7174 - val_loss: 0.5769 - val_accuracy: 0.7353
Epoch 4/10
184/184 [==============================] - 12s 66ms/step - loss: 0.5941 - accuracy: 0.7074 - val_loss: 0.5249 - val_accuracy: 0.7571
Epoch 5/10
184/184 [==============================] - 12s 66ms/step - loss: 0.6297 - accuracy: 0.6905 - val_loss: 0.5282 - val_accuracy: 0.7658
Epoch 6/10
184/184 [==============================] - 12s 66ms/step - loss: 0.6614 - accuracy: 0.6998 - val_loss: 0.6026 - val_accuracy: 0.6906
Epoch 7/10
184/184 [==============================] - 12s 66ms/step - loss: 0.7858 - accuracy: 0.6845 - val_loss: 0.8080 - val_accuracy:

In [40]:
np.mean(history2.history['val_loss'])

0.6800279736518859

In [41]:
full_test_loss = history.history['val_loss'] + history2.history['val_loss']
full_test_loss

[0.6228760480880737,
 0.5851349830627441,
 0.5699859857559204,
 0.5634013414382935,
 0.5332463979721069,
 0.6369953155517578,
 0.5470901727676392,
 0.5382905602455139,
 0.5355678200721741,
 0.5537362694740295,
 0.5876234769821167,
 0.5378829836845398,
 0.5768996477127075,
 0.5249195694923401,
 0.5282300114631653,
 0.6026430726051331,
 0.808005690574646,
 0.7587175369262695,
 1.036386489868164,
 0.8389712572097778]

In [42]:
np.mean(full_test_loss)

0.6243302315473557

answer: at leat this time the closest is 0.78, but I am skeptical cause that val_loss near the end of the second round of epochs shot up really high.

answer: and now hving run it again I get a much lower result.

again, I ran it aain (alrways reinitializing the model first) and I got that result where loss shoots up very high in the second round of epochds which I think screws the result

### Question 6

What's the average of test accuracy for the last 5 epochs (from 6 to 10)
for the model trained with augmentations?

* 0.38
* 0.58
* 0.78
* 0.98

In [43]:
np.mean(history2.history['val_accuracy'][5:])

0.6884531617164612

answer: closest is 0.78, but I dont't have that much confidence due to the varying results everytime I run it, even when trying to set up random seeds to try to attain reproducible results

this time it was closer to 0.58, which makes sense because the loss shot up this time near the end of the second round of epochs